# Train the next Crazyflie KAE
Use an Isaac teacher export for the first square KAE, or an SDK dataset export for each subsequent correction KAE. Use this repository's `KAE/` directory as the working directory. The imports below explicitly use this checkout's local runtime.

The saved dataset already declares its observation and label semantics. Do not normalize observations again or clip correction labels. The portable export contains the runtime normalizer and Koopman matrix; the training encoder remains unchanged.


In [ ]:
from pathlib import Path
import sys
import random

# Load this checkout's runtime, even if another copy is installed.
repo_root = Path.cwd().parent
if not (repo_root / "kae_moe" / "models.py").is_file():
    raise RuntimeError("Open this notebook with its KAE/ directory as the working directory.")
sys.path.insert(0, str(repo_root))

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from Autoencoder import KoopmanAutoencoder_walk
from Autoencoder_functions import compute_l_kae
from kae_moe.dataset import load_kae_dataset, export_trained_kae


## Configuration


In [ ]:
# Edit this cell for every new KAE. Paths are relative to this notebook.
dataset_dir = Path("results/square_teacher")
output_path = Path("waypoints/square_kae.pt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 10
observable_dim = 16
num_epochs = 3000
batch_size = 2048

# Keep the existing KAE architecture, loss, and one-step prediction setup.
padded_dimension = 16
kae_size = 64
p = 1
c1, c2, c3 = 1.0, 1.0, 1.0
kae_coef, action_coef = 0.1, 0.9
lr_kae, lr_kae2, lr_kae3 = 1e-3, 1e-4, 1e-5
schedule_it, schedule_it2 = 1000, 2000
clip_bound = 1.0

if observable_dim != 16 or num_epochs < 1 or batch_size < 1:
    raise ValueError("Use 16 modes and positive epoch/batch counts.")
print(f"Training on {device}; exporting to {output_path}")


## Load aligned data and initialize a fresh KAE


In [ ]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

dataset = load_kae_dataset(dataset_dir)
inputs, targets = dataset.padded_pairs()
# The existing loss helper expects one transition per sample: [N, 1, 16].
training_data = TensorDataset(inputs.unsqueeze(1), targets.unsqueeze(1))
train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=False)
aug_input_all = inputs.unsqueeze(1).to(device)
aug_output_all = targets.unsqueeze(1).to(device)
kae = KoopmanAutoencoder_walk(padded_dimension, kae_size, observable_dim, device).to(device)
optimizer_kae = torch.optim.Adam(kae.parameters(), lr=lr_kae)
kae.train()

print(f"Samples: {len(dataset.observations)}")
print(f"Input space: {dataset.metadata['training_observations']}")
print(f"Targets: {dataset.metadata['action_target']}")


## Train using the existing reconstruction, Koopman, and action losses


In [ ]:
best_loss = float("inf")
best_state = None
history = []
for epoch in (progress := tqdm(range(num_epochs), desc="Training")):
    # Updating a Python variable alone does not change Adam's learning rate.
    learning_rate = lr_kae3 if epoch > schedule_it2 else lr_kae2 if epoch > schedule_it else lr_kae
    for group in optimizer_kae.param_groups:
        group["lr"] = learning_rate
    running_loss = 0.0
    for inner, (aug_input, aug_output) in enumerate(train_loader):
        loss_kae, loss_action = compute_l_kae(
            kae, aug_input, aug_output, c1, c2, c3, p, device,
            aug_input_all, aug_output_all, inner, batch_size, 3,
        )
        loss = kae_coef * loss_kae + action_coef * loss_action
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at epoch {epoch + 1}.")
        optimizer_kae.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(kae.parameters(), max_norm=clip_bound)
        optimizer_kae.step()
        running_loss += float(loss.detach()) * len(aug_input)

        # K is fitted using encoded observations AND encoded action targets.
        # Runtime observation normalization must therefore stay out of this encoder.
        _, latent_x_all, _ = kae(aug_input_all)
        _, latent_y_all, _ = kae(aug_output_all)
        kae.compute_koopman_operator(latent_x_all, latent_y_all, device)

    average_loss = running_loss / len(training_data)
    history.append(average_loss)
    progress.set_description(f"Epoch {epoch + 1}/{num_epochs} | loss {average_loss:.5f} | lr {learning_rate:g}")
    if average_loss < best_loss:
        best_loss = average_loss
        # The external model's K is not a registered buffer: save it explicitly.
        best_state = {
            "state_dict": {key: value.detach().cpu().clone() for key, value in kae.state_dict().items()},
            "K": kae.K.detach().cpu().clone(),
            "epoch": epoch + 1,
        }

kae.load_state_dict(best_state["state_dict"])
kae.K = best_state["K"].to(device)
kae.eval()
print(f"Selected epoch {best_state['epoch']}; training loss {best_loss:.6f}")


## Validate conversion and export the portable KAE
The checkpoint includes `K`, encoder/decoder weights, a frozen runtime observation normalizer (or identity), and captured spectral bases. Isaac and the SDK load it without importing this notebook's classes. Conversion checks are numerical parity checks, not held-out controller-performance evaluation.


In [ ]:
report = export_trained_kae(
    kae, dataset, output_path,
    metadata={
        "seed": seed,
        "epochs": num_epochs,
        "selected_epoch": best_state["epoch"],
        "training_loss": best_loss,
        "mode_count": observable_dim,
    },
)
print(f"Saved {output_path.resolve()}")
print(report)
